In [5]:
from datasets import load_dataset
import pandas as pd

# Exact splits as shown in the HuggingFace viewer
splits = [
    "expert",
    "single_train",
    "single_dev",
    "multi_train",
    "multi_dev",
    "test"
]

for split in splits:
    try:
        print(f"Loading split: {split}...")
        dataset = load_dataset("Exploration-Lab/IL-TUR", "cjpe", split=split)
        
        # Convert to DataFrame
        df = dataset.to_pandas()
        
        # Save to CSV
        filename = f"cjpe_{split}.csv"
        df.to_csv(filename, index=False)
        
        print(f"✅ Saved '{filename}' — {len(df)} rows, {len(df.columns)} columns")
        print(f"   Columns: {list(df.columns)}\n")
        
    except Exception as e:
        print(f"❌ Could not load split '{split}': {e}\n")

print("Done! All 6 CJPE splits saved.")

Loading split: expert...
✅ Saved 'cjpe_expert.csv' — 56 rows, 8 columns
   Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

Loading split: single_train...
✅ Saved 'cjpe_single_train.csv' — 5082 rows, 8 columns
   Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

Loading split: single_dev...
✅ Saved 'cjpe_single_dev.csv' — 2511 rows, 8 columns
   Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

Loading split: multi_train...
✅ Saved 'cjpe_multi_train.csv' — 32305 rows, 8 columns
   Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

Loading split: multi_dev...
✅ Saved 'cjpe_multi_dev.csv' — 994 rows, 8 columns
   Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

Loading split: test...
✅ Saved 'cjpe_test.csv' — 1517 rows, 8 columns
   Columns: ['id', 'text', 'label', 'expe

In [6]:
import pandas as pd
import os

# All 6 saved CSV files
csv_files = [
    "cjpe_expert.csv",
    "cjpe_single_train.csv",
    "cjpe_single_dev.csv",
    "cjpe_multi_train.csv",
    "cjpe_multi_dev.csv",
    "cjpe_test.csv"
]

print("=" * 60)
print(f"{'CJPE DATASET SUMMARY':^60}")
print("=" * 60)

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        split_name = file.replace("cjpe_", "").replace(".csv", "")
        
        print(f"\n📂 Split : {split_name}")
        print(f"   Records    : {len(df):,}")
        print(f"   Attributes : {len(df.columns)}")
        print(f"   Columns    : {list(df.columns)}")
        print(f"   Null Values:")
        for col in df.columns:
            nulls = df[col].isnull().sum()
            print(f"      - {col}: {nulls} nulls")
        print("-" * 60)
    else:
        print(f"\n❌ File not found: {file}")

print("\n✅ Summary complete!")

                    CJPE DATASET SUMMARY                    

📂 Split : expert
   Records    : 56
   Attributes : 8
   Columns    : ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
   Null Values:
      - id: 0 nulls
      - text: 0 nulls
      - label: 0 nulls
      - expert_1: 0 nulls
      - expert_2: 0 nulls
      - expert_3: 0 nulls
      - expert_4: 0 nulls
      - expert_5: 0 nulls
------------------------------------------------------------

📂 Split : single_train
   Records    : 5,082
   Attributes : 8
   Columns    : ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
   Null Values:
      - id: 0 nulls
      - text: 0 nulls
      - label: 0 nulls
      - expert_1: 5082 nulls
      - expert_2: 5082 nulls
      - expert_3: 5082 nulls
      - expert_4: 5082 nulls
      - expert_5: 5082 nulls
------------------------------------------------------------

📂 Split : single_dev
   Records    : 2,511
   Attributes : 8

In [5]:
"""
Download NyayaFacts from HuggingFace and save all splits as CSV.
Loads each CSV file directly — bypasses HuggingFace column-consistency error.
Install:
    pip install pandas requests
"""
import pandas as pd
import requests
import os

BASE_URL = (
    "https://huggingface.co/datasets/"
    "L-NLProc/TathyaNyaya-and-FactLegalLlama-NyayaFacts-Datasets"
    "/resolve/main"
)

# ── Possible filenames per split (updated with confirmed train_single.csv) ───
SPLIT_CANDIDATES = {
    "train": [
        "train_multi.csv",
        "train_single.csv",
        "train.csv",
    ],
    "validation": [
        "dev_multi.csv",
        "dev_single.csv",
        "val_multi.csv",
        "val_single.csv",
        "validation_multi.csv",
        "validation_single.csv",
        "dev.csv",
        "val.csv",
        "validation.csv",
    ],
    "test": [
        "test_multi.csv",
        "test_single.csv",
        "test.csv",
    ],
}

# Columns expected — missing ones will be added as NaN
EXPECTED_COLUMNS = ["Case Name", "text", "label", "word_count", "Reasoning"]
LABEL_MAP = {0: "Rejected", 1: "Accepted", 2: "Multi-label"}

os.makedirs("./data", exist_ok=True)


def try_download(split_name: str, candidates: list) -> tuple[pd.DataFrame, str]:
    """Try each candidate filename; return the first successful DataFrame."""
    for filename in candidates:
        url = f"{BASE_URL}/{filename}"
        print(f"  Trying: {filename} ...", end=" ")
        try:
            resp = requests.head(url, timeout=10, allow_redirects=True)
            if resp.status_code == 200:
                print("FOUND")
                df = pd.read_csv(url, low_memory=False)
                return df, filename
            else:
                print(f"HTTP {resp.status_code}")
        except Exception as e:
            print(f"Error: {e}")
    raise FileNotFoundError(
        f"\n[ERROR] Could not find '{split_name}' split.\n"
        f"Tried: {candidates}\n"
        f"Check exact filenames at:\n"
        f"https://huggingface.co/datasets/L-NLProc/"
        f"TathyaNyaya-and-FactLegalLlama-NyayaFacts-Datasets/tree/main"
    )


def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    """Add any missing expected columns as NaN, compute word_count if possible."""

    # Compute word_count from text if missing or all-null
    if "word_count" not in df.columns or df["word_count"].isna().all():
        if "text" in df.columns:
            df["word_count"] = df["text"].apply(
                lambda x: len(str(x).split()) if pd.notna(x) else None
            )
            print(f"  'word_count' was missing — computed from 'text'.")
        else:
            df["word_count"] = None
            print(f"  WARNING: 'word_count' and 'text' both missing — set to NaN.")

    # Add any other missing expected columns as NaN
    for col in EXPECTED_COLUMNS:
        if col not in df.columns:
            df[col] = None
            print(f"  Column '{col}' was missing — added as NaN.")

    return df


# ── Download all splits ──────────────────────────────────────────────────────
all_dfs = {}

for split_name, candidates in SPLIT_CANDIDATES.items():
    print(f"\n[{split_name}] Searching ...")
    df, found_filename = try_download(split_name, candidates)
    df = normalize_df(df)

    output_path = f"./data/{split_name}.csv"
    df.to_csv(output_path, index=False)
    all_dfs[split_name] = df

    print(f"  File    : {found_filename}")
    print(f"  Saved   : {output_path}")
    print(f"  Rows    : {len(df)}")
    print(f"  Columns : {list(df.columns)}")
    if "label" in df.columns:
        label_counts = df["label"].map(LABEL_MAP).value_counts(dropna=False)
        print(f"  Labels  :\n{label_counts.to_string()}")

print("\nAll splits saved to ./data/")
print("Files:", os.listdir("./data"))


[train] Searching ...
  Trying: train_multi.csv ... FOUND
  File    : train_multi.csv
  Saved   : ./data/train.csv
  Rows    : 13629
  Columns : ['Case Name', 'text', 'label', 'word_count', 'Reasoning']
  Labels  :
label
Accepted    7523
Rejected    6106

[validation] Searching ...
  Trying: dev_multi.csv ... HTTP 404
  Trying: dev_single.csv ... HTTP 404
  Trying: val_multi.csv ... HTTP 404
  Trying: val_single.csv ... HTTP 404
  Trying: validation_multi.csv ... HTTP 404
  Trying: validation_single.csv ... HTTP 404
  Trying: dev.csv ... FOUND
  'word_count' was missing — computed from 'text'.
  File    : dev.csv
  Saved   : ./data/validation.csv
  Rows    : 1197
  Columns : ['Case Name', 'text', 'label', 'Reasoning', 'word_count']
  Labels  :
label
Rejected    629
Accepted    568

[test] Searching ...
  Trying: test_multi.csv ... HTTP 404
  Trying: test_single.csv ... HTTP 404
  Trying: test.csv ... FOUND
  'word_count' was missing — computed from 'text'.
  File    : test.csv
  Saved

In [1]:
import pandas as pd
import os

# ── Load all three splits ────────────────────────────────────────────────────
splits = {
    "train":      "./data/train.csv",
    "validation": "./data/validation.csv",
    "test":       "./data/test.csv",
}

dfs = []
for split_name, path in splits.items():
    df = pd.read_csv(path, usecols=["Case Name", "text"], low_memory=False)
    df["split"] = split_name          # optional: track which split each row came from
    print(f"[{split_name}] Rows: {len(df)}")
    dfs.append(df)

# ── Combine & save ───────────────────────────────────────────────────────────
combined = pd.concat(dfs, ignore_index=True)

output_path = "./data/combined_cases.csv"
os.makedirs("./data", exist_ok=True)
combined.to_csv(output_path, index=False)

print(f"\nCombined dataset saved : {output_path}")
print(f"Total rows             : {len(combined)}")
print(f"Columns                : {list(combined.columns)}")
print(f"\nSplit breakdown:\n{combined['split'].value_counts().to_string()}")
print(f"\nPreview:\n{combined.head(3).to_string()}")

[train] Rows: 13629
[validation] Rows: 1197
[test] Rows: 2389

Combined dataset saved : ./data/combined_cases.csv
Total rows             : 17215
Columns                : ['Case Name', 'text', 'split']

Split breakdown:
split
train         13629
test           2389
validation     1197

Preview:
                                                                                                                 Case Name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [1]:
"""
Download the IL-TUR CJPE (multi_train) dataset from HuggingFace
and save it as a clean CSV and Excel file.

Requirements:
    pip install datasets pandas openpyxl
"""

from datasets import load_dataset
import pandas as pd

# ── 1. Load the dataset ────────────────────────────────────────────────────────
print("Loading dataset from HuggingFace...")
ds = load_dataset("Exploration-Lab/IL-TUR", name="cjpe", split="multi_train")

# ── 2. Convert to a pandas DataFrame ──────────────────────────────────────────
df = pd.DataFrame(ds)

# ── 3. Preview ─────────────────────────────────────────────────────────────────
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}\n")
print(df.head(3).to_string())

# ── 4. Save as CSV (clean, UTF-8) ──────────────────────────────────────────────
csv_path = "il_tur_cjpe_multi_train.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"\n✅  Saved CSV  → {csv_path}")

# ── 5. Save as Excel (one row = one case, columns auto-sized) ─────────────────
xlsx_path = "il_tur_cjpe_multi_train.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="multi_train")

    # Auto-size columns for readability
    ws = writer.sheets["multi_train"]
    for col in ws.columns:
        max_len = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in col
        )
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 80)

print(f"✅  Saved Excel → {xlsx_path}")
print("\nDone!")

Loading dataset from HuggingFace...

Shape: 32305 rows × 8 columns

Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

       id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [5]:
"""
Download the IL-TUR CJPE (multi_train) dataset from HuggingFace
and save it as JSONL.

Requirements:
    pip install datasets pandas
"""

from datasets import load_dataset
import pandas as pd

# ── 1. Load the dataset ───────────────────────────────────────────────────────
print("Loading dataset from HuggingFace...")
ds = load_dataset("Exploration-Lab/IL-TUR", name="cjpe", split="multi_train")

# ── 2. Convert to a pandas DataFrame ─────────────────────────────────────────
df = pd.DataFrame(ds)

print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}\n")
print(df.head(3).to_string())

# ── 3. Save as JSONL (one row = one JSON line) ────────────────────────────────
jsonl_path = "cjpe_multi_train.jsonl"
df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
print(f"\n✅ Saved JSONL → {jsonl_path}")

print("\nDone!")
print("\nLoad anytime with:")
print('  df = pd.read_json("cjpe_multi_train.jsonl", lines=True)')
print('  print(df.iloc[2])  # row 3')

Loading dataset from HuggingFace...

Shape: 32305 rows × 8 columns
Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

       id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [8]:
"""
Extract only id, text, label from cjpe_multi_train.jsonl
and save as a new JSONL file.

Requirements:
    pip install pandas
"""

import pandas as pd

INPUT_PATH  = "cjpe_multi_train.jsonl"
OUTPUT_PATH = "cjpe_id_text_label.jsonl"

df = pd.read_json(INPUT_PATH, lines=True)

df[["id", "text", "label"]].to_json(OUTPUT_PATH, orient="records", lines=True, force_ascii=False)

print(f"✅ Done! {len(df)} rows saved → {OUTPUT_PATH}")
print(df[["id", "text", "label"]].head(3).to_string())

✅ Done! 32305 rows saved → cjpe_id_text_label.jsonl
       id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [3]:
"""
Download the IL-TUR CJPE (multi_train) dataset from HuggingFace
and save it as JSONL.

Requirements:
    pip install datasets pandas
"""

from datasets import load_dataset
import pandas as pd

# ── 1. Load the dataset ───────────────────────────────────────────────────────
print("Loading dataset from HuggingFace...")
ds = load_dataset("Exploration-Lab/IL-TUR", name="cjpe", split="multi_dev")

# ── 2. Convert to a pandas DataFrame ─────────────────────────────────────────
df = pd.DataFrame(ds)

print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}\n")
print(df.head(3).to_string())

# ── 3. Save as JSONL (one row = one JSON line) ────────────────────────────────
jsonl_path = "cjpe_multi_validation.jsonl"
df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
print(f"\n✅ Saved JSONL → {jsonl_path}")

print("\nDone!")


Loading dataset from HuggingFace...

Shape: 994 rows × 8 columns
Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

         id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [4]:
"""
Download the IL-TUR CJPE (multi_train) dataset from HuggingFace
and save it as JSONL.

Requirements:
    pip install datasets pandas
"""

from datasets import load_dataset
import pandas as pd

# ── 1. Load the dataset ───────────────────────────────────────────────────────
print("Loading dataset from HuggingFace...")
ds = load_dataset("Exploration-Lab/IL-TUR", name="cjpe", split="test")

# ── 2. Convert to a pandas DataFrame ─────────────────────────────────────────
df = pd.DataFrame(ds)

print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}\n")
print(df.head(3).to_string())

# ── 3. Save as JSONL (one row = one JSON line) ────────────────────────────────
jsonl_path = "cjpe_multi_test.jsonl"
df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
print(f"\n✅ Saved JSONL → {jsonl_path}")

print("\nDone!")


Loading dataset from HuggingFace...

Shape: 1517 rows × 8 columns
Columns: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']

         id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [5]:
"""
Extract only id, text, label from cjpe_multi_train.jsonl
and save as a new JSONL file.

Requirements:
    pip install pandas
"""

import pandas as pd

INPUT_PATH  = "cjpe_multi_test.jsonl"
OUTPUT_PATH = "cjpe_test_id_text_label.jsonl"

df = pd.read_json(INPUT_PATH, lines=True)

df[["id", "text", "label"]].to_json(OUTPUT_PATH, orient="records", lines=True, force_ascii=False)

print(f"✅ Done! {len(df)} rows saved → {OUTPUT_PATH}")
print(df[["id", "text", "label"]].head(3).to_string())

✅ Done! 1517 rows saved → cjpe_test_id_text_label.jsonl
        id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [6]:
"""
Extract only id, text, label from cjpe_multi_train.jsonl
and save as a new JSONL file.

Requirements:
    pip install pandas
"""

import pandas as pd

INPUT_PATH  = "cjpe_multi_validation.jsonl"
OUTPUT_PATH = "cjpe_validation_id_text_label.jsonl"

df = pd.read_json(INPUT_PATH, lines=True)

df[["id", "text", "label"]].to_json(OUTPUT_PATH, orient="records", lines=True, force_ascii=False)

print(f"✅ Done! {len(df)} rows saved → {OUTPUT_PATH}")
print(df[["id", "text", "label"]].head(3).to_string())

✅ Done! 994 rows saved → cjpe_validation_id_text_label.jsonl
        id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 